Refer to https://github.com/GreenleafLab/HDMA/blob/main/code/05-misc/04-ChromBPNet_use_cases.ipynb

In [ ]:
%load_ext autoreload
%autoreload 2

# deal with matplotlib plots
%matplotlib inline

# display all outputs in a cell
get_ipython().ast_node_interactivity = 'all'

from IPython.display import display, HTML

In [ ]:
# misc
import sys
import os
import numpy as np
import random
from tqdm import tqdm
import pandas as pd

# io
import h5py
from pyfaidx import Fasta

# ML
import bpnetlite
from bpnetlite import ChromBPNet
from bpnetlite import BPNet
from bpnetlite.bpnet import CountWrapper
from bpnetlite.attribute import deep_lift_shap

import torch

import pickle

import pyranges as pr

In [ ]:
# plotting
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn; seaborn.set_style('white')
import matplotlib.ticker as ticker
import plotnine as pn
import logomaker
import tangermeme
from tangermeme.plot import plot_logo

from tangermeme.utils import random_one_hot, _validate_input
from tangermeme.utils import characters
from tangermeme.ersatz import substitute
from tangermeme.utils import one_hot_encode

print(np.__version__)
print(tangermeme.__version__)
print(bpnetlite.__version__)
print(torch.__version__)

In [ ]:
os.environ['CUDA_VISIBLE_DEVICES']='0'
#os.environ['CUDA_VISIBLE_DEVICES']='MIG-40f43250-998e-586a-ac37-d6520e92590f'

torch.manual_seed(100)

# editable text in PDFs
# https://stackoverflow.com/a/54111532
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

sys.path.append("~/HDMA/code/03-chrombpnet")
import tangermeme_utils as tanutils
from tangermeme_utils.utils import plot_motif
from tangermeme_utils.eval import summarize_across_folds, mean_across_sequences
from tangermeme_utils.wrappers import ChromBPNetWrapper, reshape_folds

In [ ]:
torch.cuda.device_count()

In [ ]:
dev = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
dev

torch.cuda.is_available()

t1 = torch.randn(1,2).to(dev)
t1
t1.device

In [ ]:
proj_in="~/chrombpnet/chromnpnet_cardiomyocytes/peak_from_ArchR_1000bp/chrombpnet_model_training/bias_model_fibroblast05"

In [ ]:
#load model
models = [
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_0/models/chrombpnet_nobias.h5"))),
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_1/models/chrombpnet_nobias.h5"))),
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_2/models/chrombpnet_nobias.h5"))),
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_3/models/chrombpnet_nobias.h5"))),
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_4/models/chrombpnet_nobias.h5")))
]

models_counts = [
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_0/models/chrombpnet_nobias.h5"))),
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_1/models/chrombpnet_nobias.h5"))),
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_2/models/chrombpnet_nobias.h5"))),
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_3/models/chrombpnet_nobias.h5"))),
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_4/models/chrombpnet_nobias.h5")))
]

In [ ]:
#小鼠的hs3087同源序列

In [ ]:
pre_seq="TTTCCTGCCTCTCTCTAGTCACTGGGATAGTACGGTTAACCCACTGTGTACTCACTCCCTTGCCCCAAATCCGTTGAATTGTGGTACACGTGTGGAGAGTGCCGGGTGAGGGAGATGCAGATGGTGGTGATGATGATGACAGAAAGGGTGTTGGTGATGGTGATGATGGTGGTAAAAGTGATTTTGGTGCTGATGAAGGTGCTGTCAACTTCACATGGCGCTCTCTCTGAACCAGGCTCACCTCTGAGTACTGTACATATTAACTTATTGTGTCATCCCATCTCTTTTATGAGGTTCAAAAATAATTTTCACGCATCAGAACCGAAACAGGGGATGGAAATATTGGAGGCTTATACGTGTTCACTCAGAAGTCGATCGCAGTGCCTGGGAGGCTGAGGCAGGAGGCTGTTATGTTTGAAGCTAGCCTGCTCTATGACGTGGGGACTCTGTGATACAGTGCGTCAGGGGAATAGGTTACAGAGACGAAGGATAAGCTTTGTTGACATGGGAGAGACACCCACGGGTAAGTCTGGCAAATGTTCACAGTGTCCAGCAAGAAGAGCCGCTTTCTGGGCGGTGCCTCTCCTCTTCCCGTACGCCCTTCCTTATTTCCTAATCACGGCTGGGAAAAACCGAATGAACGCCAGCCACGGCCAGGAGACGCTGCGGGTGGGCGCGATGGCGATGCTGAAGCAAAGCACCCCTGACCAGCACAGCTGGACCACGACTGCACCATGAGGCAGCCCCTGGATGTCCCTCGACCTCACAGCCCCAGCCATGCAGGCAGACTGCAAAATGGTTCAAAGTGTCTGTCACATACCAGGTGCGCCCACTCTGACTCTGCCAACTTAGTGCTGCGGGAGCAGGTGTCAAACCCGATGGAATGTCATCGTCATCGGGACAACGGATGCCAAGGCCTGCCCGTTCATCAGGGGCTCTCAGGCACTTCTCTATCCGGGCCCCAGCATTCCAGGAGGACCTGCTCGCTCGGCATTATCAGCAGGCTCCGCCAGAGATAAGCAGCCTTAATGCGACCCGCTAATGGTATTATTTTAGGGACCACTCAAAGGACATGTTGGAATGGAGGCCCGGGGCCACCTGTCATTCTTTCTGTCACCGACACCCTAAACAGGTCAAACAAAGACCAGCACTTGTACACGGAGGGGGTGGGGGCATGGGAGCACGCACCTCCGAACGACTGGAAAGAATTCTCAAGCTGAAAACAGCCCCTCCCAAGAAGCAGCGGGGGCGGGGAAAAGAATGTGCTCAGAAAGCCTCCGCCCCTCACAGCACTCCTGGGTGAACAACTCCCACGCTGCAGCCAGGGTAACCGCACGCTGCAGCCAGGGTAACCGTACAGGGCGGAGGCAGAAGGCTCCTACCACGCAACTCTGTGTTGAGGAGACGTGTTAAAGCCTGCTCAGACAGACCTCTGGCCGTCTCTGGCTTCTGTTAGGAAGGGTCAGGGCTCACATGTGTAATTCTCTGCTTCCTTCAGTGGCTGAAACCATTTTTTTACGCAGCCTTAACTGTTAGTTAAGCTGAACGCTCACCATCAAACGTCAGGTGGGAAGGTGCAGACCTCTGAGCTCATGGCGGCCAGGCTCCATGCCCCGCCATCCTCCACCACCCTCCCACAGCCTCCACCGCGCTCACGTAATCATCCTCGGAGAGTATTTAGTCTGTTATTCGTGGCTTGCCTATGGCATTAAACAATGACGTGAGTTTGGGCAAGCAGGCGGCTAATGTGTCCTTCTTAGCCTCCCTAGTCACTTCCTACGAGAAAAACAGTACAGGAAACAGCTTGTGTGGAATGAAGGTCTCCGCTCCCTCAGAGCCCATGGTGTGCACATCCGTGAAACTCAGCCCGGGACCACCAGAGTCGTAGAACAAACTCCCACTGAGTTGTCAAACATTGCATGGCCTGGAAGACCAGCATGGAGGCTTTCTCCAAGGTCACCCGACTGGTTGGTGCAGCACCCACATCTGCTGACTCCCAGACGACATCTAGTCATTCAAACCCATCCAGTCAGCAGTGCTGAGTGTGAGCTTCCACTCAGTGACACAACTCGGATAACATCACTGGTCACCAGTGGCCACCTGGGCCAGCAAGC"
#使用 .upper() 转换为大写
seq_ohe = one_hot_encode(pre_seq.upper()).type(torch.float32).unsqueeze(0)

In [ ]:
# predict
print("@ predicting...")
seq_pred_list = []

for model in models:
    y_seq = tangermeme.predict.predict(model, seq_ohe)
    seq_pred_list.append(y_seq)

seq_pred = reshape_folds(seq_pred_list)

seq_pred_profile_mean, _ = summarize_across_folds(seq_pred[0])


In [ ]:
seq_pred_profile_mean.shape

In [ ]:
# compute attribution
print("@ computing attribution...")
try:

	seq_attr_list = []
	
	for model in models_counts:
		y_seq = deep_lift_shap(model, seq_ohe, random_state=0)
		seq_attr_list.append(y_seq)

	seq_attr = torch.stack(seq_attr_list, dim=0)
	seq_attr_mean, _ = summarize_across_folds(seq_attr)

	
except Exception as e:
	print(f"Error during attribution computation: {e}")

In [ ]:
results={"seq_pred_profile_mean":seq_pred_profile_mean,
    "seq_attr_mean":seq_attr_mean}

In [ ]:
pred_seq = results['seq_pred_profile_mean'].squeeze() 

In [ ]:
INPUTLEN = 2114
OUTPUTLEN = 1000
SHIFT_REL_INPUT = np.int64((2114 - 1000) / 2)
SHIFT_REL_INPUT

In [ ]:
attr_seq = results['seq_attr_mean'].sum(axis = 1).squeeze().T[(INPUTLEN // 2 - 500):(INPUTLEN // 2 + 500)]

In [ ]:
pred_seq.shape
attr_seq.shape

In [ ]:
fig, ax = plt.subplots(3, 1, figsize = (25, 6))
alpha = 0.3
# prediction in 1,000 bp
ax[0].plot(pred_seq, linewidth = 0.5, label = "seq(5-fold mean)", color = "blue")
ax[0].set_ylabel("Pred. \naccessibility")
ax[0].axvspan(273, 703, alpha = alpha, color='yellow')
ax[0].legend(fontsize = 8)
# attribution in 1,000 bp
ax[1].plot(attr_seq, linewidth = 0.5, label = "seq(5-fold mean)", color = "blue")
ax[1].set_ylabel("Contrib. \nscores")
ax[1].axvspan(273, 703, alpha = alpha, color = 'yellow')
ax[1].legend(fontsize = 8)
#
ylim = None
tangermeme.plot.plot_logo(np.float64(results['seq_attr_mean'][0]), start = 830, end = 1260, ax=ax[2], ylim = ylim)
ax[2].set_ylabel("Contrib. \n")

In [ ]:
results['seq_attr_mean'][0]

In [ ]:
results['seq_attr_mean'][0].shape

In [ ]:
import logomaker
attr_data = results['seq_attr_mean'][0].detach().cpu().numpy()
attr_data
attr_data_T = attr_data.T
attr_data_T
df = pd.DataFrame(attr_data_T, columns=['A', 'C', 'G', 'T'])
df

In [ ]:
#测试转换有无问题-----------------------------------------
# 选一段有明显波峰的区域，比如 138 到 188
test_df = df.iloc[138:188]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 6))

# 上图：用 tangermeme 画
tangermeme.plot.plot_logo(results['seq_attr_mean'][0], start=138, end=188, ax=ax1)
ax1.set_title("Tangermeme Output")

# 下图：用你的 logomaker df 画
logomaker.Logo(test_df, ax=ax2)
ax2.set_title("Logomaker Output")

plt.tight_layout()
plt.show()

In [ ]:
crp_logo = logomaker.Logo(df, 
                          figsize=(60, 2), # 2114bp 建议先设宽一点
                          shade_below=0,
                          fade_below=0)

In [ ]:
min_pos=830
max_pos=1260
df_trimmed = df.loc[min_pos:max_pos,:]

In [ ]:
crp_logo = logomaker.Logo(df_trimmed, 
                          figsize=(60, 2), # 2114bp 建议先设宽一点
                          shade_below=0,
                          fade_below=0)

In [ ]:
#Chimpanzee的hs3087

In [ ]:
pre_seq="ATTCATTATAAAAGTGAGGAAGTGTCCTCTGAGCACGCCTCCAAGGCGGTTCTGCGCAGCTCCTGCTGGGGCAGCTGAGAGTGAACGATGTTCTGAGTGAACTCCTGAGGTTCAGCAAACCTGCGCAAAGGCTCCAGGGACCCATCTTCTGAGGGGCTTCTCCTGCAGATGGTCCAGGCAGCCATCAAAGACGGGGCTGCAGGGGGGTCTGGGGCCCAGGATGACATTTGTTATTCAGTAAGTGAAGAGAGAAGGCTACAAAAATTGAGTATAATCCCTTTTGTATTAAAACAACAGCAAAACAATGTAAGTATGGATAGAAAAAAAGTCTAAAAATAAGTATAGCTAATATCCAGTGGAAGGAGAAGGTCAGATTTAAGGACAATTTTTTTTTTTTTTTTACCTTTTCCTGTGTGCCGTTCTTTATTTCCAAATCTAGGACCAAGAATAATAAATGTGAACGAAAGAAGCATTCTGATGATGAACTGTGGGCAGCCACAGAAATGCTTAAGGCACGATATTGAAGGTGAGCACCCTTCAATATCCCTGTCCCTGAAGCAGGGACAACGAGGTCACAGCTGGGCAAGGCTGGGAGGAGGAGTTCACAGGGCCCACTCTGGCGAGGACATGGCGGCAGAGGCAGCAGCAGGTGTCTGTGCACCGTGGCTTTGGTTGGGTGTCACATACCAGGTGTGCCAGGCTCTGCCGCTGCCGCCATGCCTGCACAATCAGACGTGGAGCCCATTGGACTATGTATACCCCTTGGAACATCAGCAGGCCAGCCAGGAGGCAAAGACCTTCGCAACACGTCCAAGGCCCGCAGCTGCTACCCCTCCTCACCCAGGGCTTCCGTGAGGACCTGCTCAGTGTTATCAGTGCACCCACTCCACAGATAAGAAGCTGTGCCTGGCCCCACTAATGGTATTATTTTAGGGACCACTCAAAGGACAGGCCAGGATTGGGGCCTGGGTCATCTGTCATTCCTTCCTTCATCGACACGCTAAATAAGGGTCAAACAGAGACCAGCACTTGTTTGCAAGGCTGGAGGGGCTCTGCCAAGCTGGAGTATCATCCTCCCCAGAGACTGCTGGGAAAGGCATTTGCTCAGAGCAGCTATTTCTGTCCCACTCACAGCACCTCTGGGTGACCCACCCACAGGTCAGGGTGACCTTGCAAGGTGGAAGGACAAAGGCATTGCTGCCTATCTCTTGGTGAGAGGGAATGTGTAAGATGGCTTATGCTAAAACCTCCCAAAATAGAGACACCACAGATCTGGATTCTGTTACGAAGGGTCAGAGCTCAAATATGTAATTTCTCTGCTGGCTTTAGCATTCGAAACCGTTTTTATTCAGCATTTCTTttttctttctttctttcttttttttttttcgagacagagtttcgctcttgttgcccaggctggagtgcaatggtgcaatcttggctcaccgcaacccttctgcctcccgggttcaagtgattctgctgcccctgcctcctgagtagctggcattacaggcatgtgccaccatgcccggctaattttgtagttttagtagagatgggggttctccatgttggtcaggctggtctccagctcccaacctcaggtgatcagcccacctcggcctcccaaagtgctgggattacgggcatgagccaccacacctggccCAGCATTTCTATAATTTAATTGTGTTAGATATGCACCACCGAGGGTCAAGTACAAGGATGTGGACCTTGGAGCAAATTGTGGCCAAAGTCACCACCCACCATCTGTCTCCACCTTCACCCCCACAGATACCCCCTGCCAacacacacacatgcacgcacacacacacacaTATATAAACTTTTCTTGGGAATTATTTAGTCTGTTACTCAGTGACTTGCTTGTGGCATTAAACAATGATGTGAGTTTGAGCAAACATGCATCCAAGGTCAGATTCATCACACGTAATTTCCTTCCTAAAGCCACTTCCTTTTCCAGAACGAAAAAAAGAGCAGAGGAGACAGCTACAGGGTTTGGGATGAGGGTCAGCAGGTAAGGTCACTGCTCTGTGGGGGCACGTGCTCCTGGGCACACCTGAGAACGCTTTTCTCTGATGACCCGA"
#使用 .upper() 转换为大写
seq_ohe = one_hot_encode(pre_seq.upper()).type(torch.float32).unsqueeze(0)

In [ ]:
# predict
print("@ predicting...")
seq_pred_list = []

for model in models:
    y_seq = tangermeme.predict.predict(model, seq_ohe)
    seq_pred_list.append(y_seq)

seq_pred = reshape_folds(seq_pred_list)

seq_pred_profile_mean, _ = summarize_across_folds(seq_pred[0])


In [ ]:
seq_pred_profile_mean.shape

In [ ]:
# compute attribution
print("@ computing attribution...")
try:

	seq_attr_list = []
	
	for model in models_counts:
		y_seq = deep_lift_shap(model, seq_ohe, random_state=0)
		seq_attr_list.append(y_seq)

	seq_attr = torch.stack(seq_attr_list, dim=0)
	seq_attr_mean, _ = summarize_across_folds(seq_attr)

	
except Exception as e:
	print(f"Error during attribution computation: {e}")

In [ ]:
results={"seq_pred_profile_mean":seq_pred_profile_mean,
    "seq_attr_mean":seq_attr_mean}

In [ ]:
pred_seq = results['seq_pred_profile_mean'].squeeze() 

In [ ]:
INPUTLEN = 2114
OUTPUTLEN = 1000
SHIFT_REL_INPUT = np.int64((2114 - 1000) / 2)
SHIFT_REL_INPUT

In [ ]:
attr_seq = results['seq_attr_mean'].sum(axis = 1).squeeze().T[(INPUTLEN // 2 - 500):(INPUTLEN // 2 + 500)]

In [ ]:
pred_seq.shape
attr_seq.shape

In [ ]:
fig, ax = plt.subplots(3, 1, figsize = (25, 6))
alpha = 0.3
# prediction in 1,000 bp
ax[0].plot(pred_seq, linewidth = 0.5, label = "seq(5-fold mean)", color = "blue")
ax[0].set_ylabel("Pred. \naccessibility")
ax[0].axvspan(251, 481, alpha = alpha, color='yellow')
ax[0].legend(fontsize = 8)
# attribution in 1,000 bp
ax[1].plot(attr_seq, linewidth = 0.5, label = "seq(5-fold mean)", color = "blue")
ax[1].set_ylabel("Contrib. \nscores")
ax[1].axvspan(251, 481, alpha = alpha, color = 'yellow')
ax[1].legend(fontsize = 8)
#
ylim = None
tangermeme.plot.plot_logo(np.float64(results['seq_attr_mean'][0]), start = 808, end = 1038, ax=ax[2], ylim = ylim)
ax[2].set_ylabel("Contrib. \n")

In [ ]:
results['seq_attr_mean'][0]

In [ ]:
results['seq_attr_mean'][0].shape

In [ ]:
import logomaker
attr_data = results['seq_attr_mean'][0].detach().cpu().numpy()
attr_data
attr_data_T = attr_data.T
attr_data_T
df = pd.DataFrame(attr_data_T, columns=['A', 'C', 'G', 'T'])
df

In [ ]:
#测试转换有无问题-----------------------------------------
# 选一段有明显波峰的区域，比如 138 到 188
test_df = df.iloc[138:188]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 6))

# 上图：用 tangermeme 画
tangermeme.plot.plot_logo(results['seq_attr_mean'][0], start=138, end=188, ax=ax1)
ax1.set_title("Tangermeme Output")

# 下图：用你的 logomaker df 画
logomaker.Logo(test_df, ax=ax2)
ax2.set_title("Logomaker Output")

plt.tight_layout()
plt.show()

In [ ]:
crp_logo = logomaker.Logo(df, 
                          figsize=(60, 2), # 2114bp 建议先设宽一点
                          shade_below=0,
                          fade_below=0)

In [ ]:
min_pos=808
max_pos=1038
df_trimmed = df.loc[min_pos:max_pos,:]

In [ ]:
crp_logo = logomaker.Logo(df_trimmed, 
                          figsize=(60, 2), # 2114bp 建议先设宽一点
                          shade_below=0,
                          fade_below=0)

In [ ]:
INPUTLEN = 2114
OUTPUTLEN = 1000
SHIFT_REL_INPUT = np.int64((2114 - 1000) / 2)
SHIFT_REL_INPUT

In [ ]:
def predict_species_sequence(pre_seq,core_start_input,core_end_input,core_start_output,core_end_output,species,region,show_fig=True):

    #使用 .upper() 转换为大写
    seq_ohe = one_hot_encode(pre_seq.upper()).type(torch.float32).unsqueeze(0)

    # predict
    print("@ predicting...")
    seq_pred_list = []

    for model in models:
        y_seq = tangermeme.predict.predict(model, seq_ohe)
        seq_pred_list.append(y_seq)

    seq_pred = reshape_folds(seq_pred_list)

    seq_pred_profile_mean, _ = summarize_across_folds(seq_pred[0])

    seq_pred_profile_mean.shape

    # compute attribution
    print("@ computing attribution...")
    try:

        seq_attr_list = []
        
        for model in models_counts:
            y_seq = deep_lift_shap(model, seq_ohe, random_state=0)
            seq_attr_list.append(y_seq)

        seq_attr = torch.stack(seq_attr_list, dim=0)
        seq_attr_mean, _ = summarize_across_folds(seq_attr)

        
    except Exception as e:
        print(f"Error during attribution computation: {e}")

    results={"seq_pred_profile_mean":seq_pred_profile_mean,
        "seq_attr_mean":seq_attr_mean}

    pred_seq = results['seq_pred_profile_mean'].squeeze()
    area_seq = pred_seq.sum()      # 计算预测序列开放性曲线下的总面积

    attr_seq = results['seq_attr_mean'].sum(axis = 1).squeeze().T[(INPUTLEN // 2 - 500):(INPUTLEN // 2 + 500)]

    print(f"Accessibility Prediction score area: {area_seq}")
    # ploting-1
    fig, ax = plt.subplots(3, 1, figsize = (25, 6))
    alpha = 0.3
    # prediction in 1,000 bp
    ax[0].plot(pred_seq, linewidth = 0.5, label = "seq(5-fold mean)", color = "blue")
    ax[0].set_ylabel("Pred. \naccessibility")
    ax[0].axvspan(core_start_output, core_end_output, alpha = alpha, color='yellow')
    ax[0].legend(fontsize = 8)
    # attribution in 1,000 bp
    ax[1].plot(attr_seq, linewidth = 0.5, label = "seq(5-fold mean)", color = "blue")
    ax[1].set_ylabel("Contrib. \nscores")
    ax[1].axvspan(core_start_output, core_end_output, alpha = alpha, color = 'yellow')
    ax[1].legend(fontsize = 8)
    #
    ylim = None
    tangermeme.plot.plot_logo(np.float64(results['seq_attr_mean'][0]), start = core_start_input, end = core_end_input, ax=ax[2], ylim = ylim)
    ax[2].set_ylabel("Contrib. \n")
        # put a title at the very top
    plt.suptitle(species + "_"+ region, fontsize = 12)
    plt.xlabel("Genomic position (bp)")
    plt.tight_layout()
    
    if show_fig:
        plt.show()
    else:
        plt.savefig(os.path.join(outdir, species + "_"+ region + ".pdf"))

    plt.close(fig)

    
    ## ploting-2
    import logomaker
    attr_data = results['seq_attr_mean'][0].detach().cpu().numpy()
    attr_data_T = attr_data.T
    df = pd.DataFrame(attr_data_T, columns=['A', 'C', 'G', 'T'])
    crp_logo = logomaker.Logo(df, 
                          figsize=(120, 6), # 2114bp 建议先设宽一点
                          shade_below=0,
                          fade_below=0)
    ### ploting-3
    min_pos=core_start_input
    max_pos=core_end_input
    df_trimmed = df.loc[min_pos:max_pos,:]
    crp_logo = logomaker.Logo(df_trimmed, 
                            figsize=(25, 2), # 2114bp 建议先设宽一点
                            shade_below=0,
                            fade_below=0)


In [ ]:
outdir="~/chrombpnet/chromnpnet_cardiomyocytes/peak_from_ArchR_1000bp/data/region_for_predict/hs3087"

In [ ]:
#zebrafish
species="zebrafish"
region="hs3087-1"
pre_seq="acatgttttatgagtgtgttgataagcgaagtttgatttaaatcctacacggcgcctcatgatatatttttgtttaaacacgcacccactttcgctccgtttcaaatcacatatacagagtctgtttgggaaaaagagtctatttaacactataaaacgcgtcagctgacagtcatgcaccattatatgactgttttgaggattgcattggaggggtgacggcacctgtaatgaaaaggaaagggaattgtgccgttttcatatttatttcaaagtgttttcgtgcctaaacaaagcaaaagcacagaagagactcacatttaggagcaaggggcggcccctagtggttggggggtatgggttgcatatagggaggctcgactcaaatgtttcttgccacccttcagagaaacactgaaattaacgggagaaatacgggcaaatacctttacgggatgatagtgggaaaaaactgcaaaatacaggagaatcctgggaaaaacgggaggtttgacatgtatgctcaacacacacaaatctgttaaaatctctaaaaaagttcttgagggtttcgtgaatctttaagcacaatagacatttttcctgacaaaacaaaaatagatttttattaataaaaaaacagacacctcatacccaccttcaaattttgtttgtgtgtgtgtgtgtgtctatatatgtgtgtgcgcgccttctcttggactggcagaggtcaaagatcagctcttgaggtggaaataaatactctccgtttctccttctcctcttcagggtcgggatggagcaggagacccagaagagcaggtcataaaacaacatgacaccaagaggaagtccacgttcggcgtgctgaaagatgacagcgcttgcagactgagatcttaatgcttttttgggagttggcgaaggggtcgggggaccgagaggagatttttccctcttagtcccacattaccaccccattaacctataacccgatgttgtggagtgcagcggagcttcagtctgaggcacatgagtggaaagtagagaaagtggggagcacggcccatgacctcgcacacacacacacgcacacacacatggtctgcatttgggtgtctgcgggtggcatggggctaacgctaatctgaaacagaatctgctataaacaggatgttgagagtggacacaaagagaaagcagaaatgaggtgagcaggaggaagagaaggaggtcttggtaagggtttgggtttggtggtctaaatacaaaaagtcaactgttattttcaattttacagcgactttcttgtaattgagatgttctcacctcacaatgtgattttaaatgtcacaagtgtggttttatgggctacatgcacctggacttttaatttctagtaagccagctcaagctagcaaagcatcagttacagtcaggtccataaatattgggctctatatatttaaatataaatattttggctctatagcacacacacctggccatgaaacaactaagaagccaattgtcccataacttttggtcccttaacaagtgagaggcacaaatgcaaactgttgtaattcctacaccgttgacccgatttgaatgtaaatactctcaaattgaagctgacagtctgcagttaaagcactagagctacacgattctggctgaaatgagaattgcgattttttttttgcttaaaataaagagcaccatttttacgattctgtagatgtaaaataaaggttaatatgagtcggcaatttttatagttttttttttttcaaacatggtaaaacaacagtaataatatgtgtaagtctactggtttttataaataattctattctacttataataaaacgagttaaatctaatcgagattgcaatattattttatttattgtgcagctctataaagcacatcttcatttcatttcaaatccattgtagagccaaaaatgttagaattgtgtcgatgtcccaatattatttagacctgactctatgtattgactataggctttcaaagtttttcaacgtgtgtgtgtgtgtgtgtttgccggtcctacgccacccaaccaacccgctccgagcttttatca"
core_start_input=762
core_end_input=1311
core_start_output=core_start_input-SHIFT_REL_INPUT
core_end_output=core_end_input-SHIFT_REL_INPUT
predict_species_sequence(pre_seq,core_start_input,core_end_input,core_start_output,core_end_output,species,region,show_fig=True)

In [ ]:
#zebrafish
species="zebrafish"
region="hs3087-2"
pre_seq="atgctctagactagtttttaacagcagatgatgctctaggctagtttttaacagcagatgatgctctagactagtttttttaacagcagatggagctttaggctagtttttaggaatagatggtgctctagactagttttaatagaagatgatgctctaggctagcgattaacaacagatagatggtgggctaggctaggctttaaaagcagatggtgctctttaggttagtttttaactctacactcaaatcttcaagaagaagagtctaggtcatttgttcaaatgtactgcacctcagcttttatgccatgggattaatataaacactgctcactatgaactctcttgtaattctgtcaccccaaatttgaaatgtacacttggaaattatagaaaattctatataatttatagtttgtcaagataagctcaggtttaaaaaaagaaatgtaaaaatgtaataaaagcatacaatgcctaaacatttgcataacttcataaatggttattttaggttcaggtggtacgagacagagtatggcagatagttcagagtcagaattgaatcaacagaaaaatcgcaatgcattctgaaaatcaaactcatattatctcaatgcatcgatttattgccccagccctaatcctaattcaccccaaatagtattgtcttaatcatcttaatggcagaatattgttatccatgtaaacatgctgtgtaaaaacttcaaaagtgaatcagaggtaatataaagccttctgttatagcgagggtctctgagtcccgtccagctgttcagagccagtgtttatgtgtgtgatccagcagaagagacagatctgctacaggtggcaggactgagggtgtgtaaacagaaaggaggctctgctgtctctgctcgtgctcttgaggtaatgcagcctaatggagctacacacactcataaaactgcttgagccataaacacacagagagataaagaggaatgtggagtgagatggagaaggaccagagctaacagaagaaaactgctgggggaataaaagttttacagattatgaagaagatttctattctctctgtgtgaagtattttctaagtgtaaagcagttaattttctcttctcaaaatgtatgtagtgtttgcagcaaacatgccagcctgtgttaaaaatacagtgatatgtctgtgtaagatgagagtgcctgcacagaagcaacactaggagaaggctgagtgactgattgtgtagaataactgaatatgtaaagcactgattatatatgttaaatacttctatgaagctatatgtgtcagaagttagaatgagtaacatattatcaagttattatgctttcatgacaatgtatgtaaacacttagttcctttgcataagttgcatctatactttgctgacatagacctgtaacatctctgttgacatgagctagtgggctgagagccaagaatagcaggaccccttagactaaaaacctattcaaaaatggtcaagagttaaaaagagacttaggggtgtgtcaaataatcctgtataaaaattgtaacaaacacagatacttcagaaggtgcaggagagatttcagaaggtgtccaggagagacttcagaatggtctggggtctgctctgctctttctctgctttattatggagaataaagtctattttctgatattcaaaacctccggtctgataatttgtaattaatgagaagtcaatatttaccacaattacaatggcttttgggatgaaaaaatcttctctccgttaaacagaaattgtggggaaaaataaatgagggggctaataattctgacttcaactgtatatgttttttcaagtgtcttgtagaaaatattaatttaaaagagagatctgtgagagatgtactcaattatactgagcagcagtaaaaaaataacaaatatacagtagtggacagagttctgaaaaatcatacttaagttaaagtaccattacttgcttaagtagagtaaaagtatctgttgtaaatattactcaaactatcagtaaaaagaagacttttcaaaagtactcaagagtagtgagtattaccctgtgaaaagctga"
core_start_input=762
core_end_input=1311
core_start_output=core_start_input-SHIFT_REL_INPUT
core_end_output=core_end_input-SHIFT_REL_INPUT
predict_species_sequence(pre_seq,core_start_input,core_end_input,core_start_output,core_end_output,species,region,show_fig=True)

In [ ]:
#mouse
species="mouse"
region="hs3087"
pre_seq="TTTCCTGCCTCTCTCTAGTCACTGGGATAGTACGGTTAACCCACTGTGTACTCACTCCCTTGCCCCAAATCCGTTGAATTGTGGTACACGTGTGGAGAGTGCCGGGTGAGGGAGATGCAGATGGTGGTGATGATGATGACAGAAAGGGTGTTGGTGATGGTGATGATGGTGGTAAAAGTGATTTTGGTGCTGATGAAGGTGCTGTCAACTTCACATGGCGCTCTCTCTGAACCAGGCTCACCTCTGAGTACTGTACATATTAACTTATTGTGTCATCCCATCTCTTTTATGAGGTTCAAAAATAATTTTCACGCATCAGAACCGAAACAGGGGATGGAAATATTGGAGGCTTATACGTGTTCACTCAGAAGTCGATCGCAGTGCCTGGGAGGCTGAGGCAGGAGGCTGTTATGTTTGAAGCTAGCCTGCTCTATGACGTGGGGACTCTGTGATACAGTGCGTCAGGGGAATAGGTTACAGAGACGAAGGATAAGCTTTGTTGACATGGGAGAGACACCCACGGGTAAGTCTGGCAAATGTTCACAGTGTCCAGCAAGAAGAGCCGCTTTCTGGGCGGTGCCTCTCCTCTTCCCGTACGCCCTTCCTTATTTCCTAATCACGGCTGGGAAAAACCGAATGAACGCCAGCCACGGCCAGGAGACGCTGCGGGTGGGCGCGATGGCGATGCTGAAGCAAAGCACCCCTGACCAGCACAGCTGGACCACGACTGCACCATGAGGCAGCCCCTGGATGTCCCTCGACCTCACAGCCCCAGCCATGCAGGCAGACTGCAAAATGGTTCAAAGTGTCTGTCACATACCAGGTGCGCCCACTCTGACTCTGCCAACTTAGTGCTGCGGGAGCAGGTGTCAAACCCGATGGAATGTCATCGTCATCGGGACAACGGATGCCAAGGCCTGCCCGTTCATCAGGGGCTCTCAGGCACTTCTCTATCCGGGCCCCAGCATTCCAGGAGGACCTGCTCGCTCGGCATTATCAGCAGGCTCCGCCAGAGATAAGCAGCCTTAATGCGACCCGCTAATGGTATTATTTTAGGGACCACTCAAAGGACATGTTGGAATGGAGGCCCGGGGCCACCTGTCATTCTTTCTGTCACCGACACCCTAAACAGGTCAAACAAAGACCAGCACTTGTACACGGAGGGGGTGGGGGCATGGGAGCACGCACCTCCGAACGACTGGAAAGAATTCTCAAGCTGAAAACAGCCCCTCCCAAGAAGCAGCGGGGGCGGGGAAAAGAATGTGCTCAGAAAGCCTCCGCCCCTCACAGCACTCCTGGGTGAACAACTCCCACGCTGCAGCCAGGGTAACCGCACGCTGCAGCCAGGGTAACCGTACAGGGCGGAGGCAGAAGGCTCCTACCACGCAACTCTGTGTTGAGGAGACGTGTTAAAGCCTGCTCAGACAGACCTCTGGCCGTCTCTGGCTTCTGTTAGGAAGGGTCAGGGCTCACATGTGTAATTCTCTGCTTCCTTCAGTGGCTGAAACCATTTTTTTACGCAGCCTTAACTGTTAGTTAAGCTGAACGCTCACCATCAAACGTCAGGTGGGAAGGTGCAGACCTCTGAGCTCATGGCGGCCAGGCTCCATGCCCCGCCATCCTCCACCACCCTCCCACAGCCTCCACCGCGCTCACGTAATCATCCTCGGAGAGTATTTAGTCTGTTATTCGTGGCTTGCCTATGGCATTAAACAATGACGTGAGTTTGGGCAAGCAGGCGGCTAATGTGTCCTTCTTAGCCTCCCTAGTCACTTCCTACGAGAAAAACAGTACAGGAAACAGCTTGTGTGGAATGAAGGTCTCCGCTCCCTCAGAGCCCATGGTGTGCACATCCGTGAAACTCAGCCCGGGACCACCAGAGTCGTAGAACAAACTCCCACTGAGTTGTCAAACATTGCATGGCCTGGAAGACCAGCATGGAGGCTTTCTCCAAGGTCACCCGACTGGTTGGTGCAGCACCCACATCTGCTGACTCCCAGACGACATCTAGTCATTCAAACCCATCCAGTCAGCAGTGCTGAGTGTGAGCTTCCACTCAGTGACACAACTCGGATAACATCACTGGTCACCAGTGGCCACCTGGGCCAGCAAGC"
core_start_input=762
core_end_input=1311
core_start_output=core_start_input-SHIFT_REL_INPUT
core_end_output=core_end_input-SHIFT_REL_INPUT
predict_species_sequence(pre_seq,core_start_input,core_end_input,core_start_output,core_end_output,species,region,show_fig=True)

In [ ]:
#黑猩猩
species="chimpanzee"
region="hs3087"
pre_seq="ATTCATTATAAAAGTGAGGAAGTGTCCTCTGAGCACGCCTCCAAGGCGGTTCTGCGCAGCTCCTGCTGGGGCAGCTGAGAGTGAACGATGTTCTGAGTGAACTCCTGAGGTTCAGCAAACCTGCGCAAAGGCTCCAGGGACCCATCTTCTGAGGGGCTTCTCCTGCAGATGGTCCAGGCAGCCATCAAAGACGGGGCTGCAGGGGGGTCTGGGGCCCAGGATGACATTTGTTATTCAGTAAGTGAAGAGAGAAGGCTACAAAAATTGAGTATAATCCCTTTTGTATTAAAACAACAGCAAAACAATGTAAGTATGGATAGAAAAAAAGTCTAAAAATAAGTATAGCTAATATCCAGTGGAAGGAGAAGGTCAGATTTAAGGACAATTTTTTTTTTTTTTTTACCTTTTCCTGTGTGCCGTTCTTTATTTCCAAATCTAGGACCAAGAATAATAAATGTGAACGAAAGAAGCATTCTGATGATGAACTGTGGGCAGCCACAGAAATGCTTAAGGCACGATATTGAAGGTGAGCACCCTTCAATATCCCTGTCCCTGAAGCAGGGACAACGAGGTCACAGCTGGGCAAGGCTGGGAGGAGGAGTTCACAGGGCCCACTCTGGCGAGGACATGGCGGCAGAGGCAGCAGCAGGTGTCTGTGCACCGTGGCTTTGGTTGGGTGTCACATACCAGGTGTGCCAGGCTCTGCCGCTGCCGCCATGCCTGCACAATCAGACGTGGAGCCCATTGGACTATGTATACCCCTTGGAACATCAGCAGGCCAGCCAGGAGGCAAAGACCTTCGCAACACGTCCAAGGCCCGCAGCTGCTACCCCTCCTCACCCAGGGCTTCCGTGAGGACCTGCTCAGTGTTATCAGTGCACCCACTCCACAGATAAGAAGCTGTGCCTGGCCCCACTAATGGTATTATTTTAGGGACCACTCAAAGGACAGGCCAGGATTGGGGCCTGGGTCATCTGTCATTCCTTCCTTCATCGACACGCTAAATAAGGGTCAAACAGAGACCAGCACTTGTTTGCAAGGCTGGAGGGGCTCTGCCAAGCTGGAGTATCATCCTCCCCAGAGACTGCTGGGAAAGGCATTTGCTCAGAGCAGCTATTTCTGTCCCACTCACAGCACCTCTGGGTGACCCACCCACAGGTCAGGGTGACCTTGCAAGGTGGAAGGACAAAGGCATTGCTGCCTATCTCTTGGTGAGAGGGAATGTGTAAGATGGCTTATGCTAAAACCTCCCAAAATAGAGACACCACAGATCTGGATTCTGTTACGAAGGGTCAGAGCTCAAATATGTAATTTCTCTGCTGGCTTTAGCATTCGAAACCGTTTTTATTCAGCATTTCTTttttctttctttctttcttttttttttttcgagacagagtttcgctcttgttgcccaggctggagtgcaatggtgcaatcttggctcaccgcaacccttctgcctcccgggttcaagtgattctgctgcccctgcctcctgagtagctggcattacaggcatgtgccaccatgcccggctaattttgtagttttagtagagatgggggttctccatgttggtcaggctggtctccagctcccaacctcaggtgatcagcccacctcggcctcccaaagtgctgggattacgggcatgagccaccacacctggccCAGCATTTCTATAATTTAATTGTGTTAGATATGCACCACCGAGGGTCAAGTACAAGGATGTGGACCTTGGAGCAAATTGTGGCCAAAGTCACCACCCACCATCTGTCTCCACCTTCACCCCCACAGATACCCCCTGCCAacacacacacatgcacgcacacacacacacaTATATAAACTTTTCTTGGGAATTATTTAGTCTGTTACTCAGTGACTTGCTTGTGGCATTAAACAATGATGTGAGTTTGAGCAAACATGCATCCAAGGTCAGATTCATCACACGTAATTTCCTTCCTAAAGCCACTTCCTTTTCCAGAACGAAAAAAAGAGCAGAGGAGACAGCTACAGGGTTTGGGATGAGGGTCAGCAGGTAAGGTCACTGCTCTGTGGGGGCACGTGCTCCTGGGCACACCTGAGAACGCTTTTCTCTGATGACCCGA"
#core_start_input=737#short
core_start_input=637 #long
core_end_input=1186
core_start_output=core_start_input-SHIFT_REL_INPUT
core_end_output=core_end_input-SHIFT_REL_INPUT
predict_species_sequence(pre_seq,core_start_input,core_end_input,core_start_output,core_end_output,species,region,show_fig=True)

In [ ]:
#pig
species="pig"
region="hs3087"
pre_seq="ttccacggcgccacgtcgggaactccACAATCACATAATCTTGATTGTGTTTCCCTTGCCCAAGGGTTGTGGGGACAAGGCTGATGGAAACTCCCCATCATCCTGTCTCTGCAGAAATACAAGGAGCCTTCTCTGAAAACCTGTCCCACAGTGGCCAAAGGCACCACCACCGCCCCATGAATGTTGACCGTGTCGCTGTCTCTGCTGGGGCCGCGTCCCCTGCCTGCTGGGTGGTCTCTTGCTTGTCTGGCTCCGCCGGACTGTCTCACCCTTCTCGCCCCCCGGCCCGTGAGTGCGGGAGAGCAGGTTCCTGTCCACCTGCCTCTCTGTTTCCAGGACAGCGCCTGGCGCAGAGGCAGAGCCGAAGGCGTCCGTGCGGGTGTCGGCCGCCAGGGGATGCAGCCACAGAGCAGAGTGGAACTGTGCCGAGTGGTCACAGCCCGGGGGAGGAGCACACTGGATTGATGGGTGATTGTTCACCTTTTCCATGTGCGGTTCTTTATTTCCAAATCTAGCATCGGGAATAATCGACCTTTGCAAAGGTCGAGGACGAGGCGAGGACAGCCAGAACGGGGCCTGAAGAGTGACCTCGAAGGTGAGCTCGCCGCAGCCAACGGCAGGGACAGGAAGTCACAGCTGGGAAGATGTGCCCACCGGGTGAGCCCCCCACCCCGTGACGTCCCCGGGTCCACAGGCCCCGTCTGGATGGGGACAGGTGGGCAGGAGCCGTGGTGGGTCTCGGTACCCACGTGACTCTTGGTGGGTGTCATGGACCCTGCCACCAAGGTCACGCCTGCACACGCAGGCCTCCAGACCGCCAGACTATTTATACCCCTGGAATGTCACCAGGTCACCGGGAAGCAAAGACTTTTGCAACACGTCAAGGCCACGGCCACCGCCCCGTCTCGGCCCGGGGCATCCGAGAGGCCCCACTCAGCGTTATCAGCAAGCCCCTCTGAAGATAAGGAGCCTCCGCTTGGCCCCACTAATGGTATTATTTTAGGGACCACTCAGAAAGACAGGACGGGATGGAGGCCCAGGTCATCTGTCATTCCTTCCCTCATCAACACGCAGAACAAGAACGGAACAAAGACCAGGGCTCGTTTGGGGGGCGGGGGGAGGAGGCTCCACCGAGGGGAATATCAGCCCCCCAAAGGCACCTGGGAAAGGAATTTGCTCAGAACGACCCTCCTTCCCCCTCATAGCGCTTCTGGGCAACCCACCCACCCACCAGCCAAGGTGCCTTGCCAGCTGGAGGGGCTGGACCAAAGAAGTAGGATGACTGCGGCTGAAATGTCCTTGACAGAGAGATGACGGATCTGGATTCAGTTAGAAAGGGTCAGAGCTCAAATATGTCATTTCTCCGTCGGCTTTAGCCATTGAAACCAATTGTATTCAGCATCTCTCTAACTTGCTTAAGTTGGATGTGTACCGTCAAGGGTCAGGGACAAGGGTGTGGGCCTCGGAGCAAACTGAGGCAGCGTCTCCTCTGcccccacctccacccccacccccacgcccacgcccacacccacACGCGCGCGCTCTCTCCTGGGAATTACTTTGTTGCTTGGCAGCTCGCTCGCCATCAAATAATGACGTGTGTTTGGGAAAACACGGATCTAAGTCAGGGTTATTGCTCCTAGTGCCCCAGCCACTTCCCTTCCTAGACCAGGAAAAGTGAAGACGACTGCCGGGTTTGAGGTGAAAGTCAGAGGGTCACTTGGCTGCTGCACAAAAGCACGTTTCTGGGTACACATGGCCCCCAAATGCTACCCAGATGCCAAAGGGGCCGGTCAGGTAGAGCAGACAGCGAGCCCGCAGCCCAGCCTTGATCTGCAGGGCAGGGGTCAGGGGCTGGGGGCTGAGCCCTCTCCAAGGCCCCCAGAGGCAGAGCCCTAGTCTCCTGATTCCGGGCTCGTGAATGACAGTACAGGCATTTATTCATGGCATCCCGTCAAGGAAGCTctgctgcgggccaagccttgggctgggccctggagaccccgcagcaaccagaacagacaaatccctccggagggccggatggaggagagcccacaggtaaccaacacattccgtttgccaggtggccaggagcttgatggaggaag"
core_start_input=710
core_end_input=1259
core_start_output=core_start_input-SHIFT_REL_INPUT
core_end_output=core_end_input-SHIFT_REL_INPUT
predict_species_sequence(pre_seq,core_start_input,core_end_input,core_start_output,core_end_output,species,region,show_fig=True)

In [ ]:
#rabbit
pre_seq="ccccacccccgccttgtGTCCCGCCCCTACTCTGACAGCTCCAACCAGCAACCTCACGGTGCCCTGAACAACACACACAGGTTCTCCCCTCCTCAGAATTCTAGGAAGCGCTGGCACCTTCGGAAGGGCTGACCTCCCAGAGCCCACAGGTCCCGCACTCCTGGCTcacagcccctgcctccagcGCAGTGCCTCCTGGTCTCGCAGACCTGGGTTCAGGCCGATCCGCTGTTCTCCGATCGCGGATGGCTGCCCGCTGCTTCCTTGGTGCCCTGGGGGCTTGGTTCCAAGTTGCCCCGGGCACTGGTCCTGCCCGGAgccctgctggctcctcccagctCTCCTGTCCCCTTAGCCCCTCGAGTGCTCCTGCGGCTCCAAATCCAAGCTCAGGGGCCGGAGCGACGACGGGGGAGGCGCGCCCTGGAGCGGGCTCCGGTGGAGCAGTGACCTCTCCCGCTGGCCTTCTTTCCCAACCCTGCAGCTGCCGCCTCCGCCATCTCCACTCGGGGAGAGGAAGTGGCCGGGGGAAGCGAAGAATGAAGAGTCTGACCTTAGCCGTGTGTTTGCCCAAACCCACGTCATTGTTTAATGCCACCAGCGAGCAGCCCAGGAACAGACTAAATAATCCCAGGACAAGtttatgtgtgtggggggtggaggTGAGGCGGATGTGGGCAGAGACGTGGCCCACAGCGTGCTCAGAGGCCCACAGCCTGGAGGGCCACCTTGGTGGTGCACGTGCCGCTCAGTGACGTAGAAATCTGAACAAAATGGTTTTGGCGGCTCAAGCCAGCGGAGAACCGACGTCGAAGGTTTAGGAGGTTTTAGGAGGCTTCGGCACAGGTCATATCCACATTCCCACCTCGCAAGGTCGCTCGGGCCTGCGGCCTGCGGCTGGGAGGTTCCGGGGAGGGGGCTGGAAACGGCCGTTCCGGGCAGTCTTTTCTAGCTGCCTCCCGGGAGGGCCCCTCAGACAAGGGCTGGTCTTTGTTCGGCCCTGGGTTAGTTCCTTGATGAAGGAGAGGATGACAGATGGCTGGGCCCGCATCCCAGCCCGTCCGCTGAGTGGTCCCTAAAATAATACCATTAGTGGGTCAGGCCGAGGCTCCTTATCTGCGGAGTGGGCCTGCTGATAAGGCGGAGCAGGGTCCCCCAGAAGCCCGGGCCGCGGATGAATTGCACAGTTCTTTGCTTCCGGCTGGCTTCTGACATTCCAAGGGGTAAAAATAGCTCGAGGGCCGGACACCCGGCTGTGCCAACGAGACGTCTGCCATGCAGCAGGTGCACACCACTCACCTTAGGCCACATGTGCGGAGACCAGCTGCCCCTTCTGTCCACGTGTCCTCAGCCTGAGCCAGGACCTGCGAGCTCTGGGGACACCGAGGGGACTCCTCCTGTGTCCCCTCTGTCGGCTGTGGCCACTCACCTTGGAGGTCGCCGTCAGACATTCAGGCTGCCCGCAGGTCATCACCACCGGGCTCATTTGAGTcttactcctgactccagatttggAAATAAAGAATGGGCCCCTCGGAGAGGAAAGAAATTGCCCCGAAACCCAACCTCCCACTGACGCAAAATGCTACGCCTATCTGCCACGCCTATTTCTGGATTTTCCCCACCTACGCTCGTGCTGTTTTGTTGATGTTTGTGTCCcggggcccagcccctgcagcaccACCCTGGACGGGCGCACGCGTGGGCACCAGGCTGCCGGGTCCCATCCCCCGCCATCTCCcgtcagctgtgtggccttggctgCTCAGCTcttttccgggtctctcacgtgtcTCATGTCTCATGTCTCTCACGTGTCTCATGTCTCTCACGTGTGAGGTGGGCTTGTGCGTGTTCCTCACCTCTCAGGAGCATAGCGAAGATCAAATGAGAGTGCTTAGGAGTGCGCCTGGTTCACAATGAGCAGAGAAAGTGTTCATCgtcaacaccaccaccaccatcaccatcaccatcaccaccaccatcaccaccaccaccattatcaccaccaccaccatcaccaccatcatcatcttcatcaccaccaccatcaccatcactaccaccattatcaccaccatcaccaccacaattatcaccaccatcaccatcac"
core_start_input=805
core_end_input=1363
core_start_output=core_start_input-SHIFT_REL_INPUT
core_end_output=core_end_input-SHIFT_REL_INPUT
predict_species_sequence(pre_seq,core_start_input,core_end_input,core_start_output,core_end_output)

In [ ]:
#rabbit_rev---用这个
species="rabbit"
region="hs3087"
pre_seq="gtgatggtgatggtggtgataattgtggtggtgatggtggtgataatggtggtagtgatggtgatggtggtggtgatgaagatgatgatggtggtgatggtggtggtggtgataatggtggtggtggtgatggtggtggtgatggtgatggtgatggtggtggtggtgttgacGATGAACACTTTCTCTGCTCATTGTGAACCAGGCGCACTCCTAAGCACTCTCATTTGATCTTCGCTATGCTCCTGAGAGGTGAGGAACACGCACAAGCCCACCTCACACGTGAGAGACATGAGACACGTGAGAGACATGAGACATGAgacacgtgagagacccggaaaagAGCTGAGcagccaaggccacacagctgacgGGAGATGGCGGGGGATGGGACCCGGCAGCCTGGTGCCCACGCGTGCGCCCGTCCAGGGTggtgctgcaggggctgggccccgGGACACAAACATCAACAAAACAGCACGAGCGTAGGTGGGGAAAATCCAGAAATAGGCGTGGCAGATAGGCGTAGCATTTTGCGTCAGTGGGAGGTTGGGTTTCGGGGCAATTTCTTTCCTCTCCGAGGGGCCCATTCTTTATTTccaaatctggagtcaggagtaagACTCAAATGAGCCCGGTGGTGATGACCTGCGGGCAGCCTGAATGTCTGACGGCGACCTCCAAGGTGAGTGGCCACAGCCGACAGAGGGGACACAGGAGGAGTCCCCTCGGTGTCCCCAGAGCTCGCAGGTCCTGGCTCAGGCTGAGGACACGTGGACAGAAGGGGCAGCTGGTCTCCGCACATGTGGCCTAAGGTGAGTGGTGTGCACCTGCTGCATGGCAGACGTCTCGTTGGCACAGCCGGGTGTCCGGCCCTCGAGCTATTTTTACCCCTTGGAATGTCAGAAGCCAGCCGGAAGCAAAGAACTGTGCAATTCATCCGCGGCCCGGGCTTCTGGGGGACCCTGCTCCGCCTTATCAGCAGGCCCACTCCGCAGATAAGGAGCCTCGGCCTGACCCACTAATGGTATTATTTTAGGGACCACTCAGCGGACGGGCTGGGATGCGGGCCCAGCCATCTGTCATCCTCTCCTTCATCAAGGAACTAACCCAGGGCCGAACAAAGACCAGCCCTTGTCTGAGGGGCCCTCCCGGGAGGCAGCTAGAAAAGACTGCCCGGAACGGCCGTTTCCAGCCCCCTCCCCGGAACCTCCCAGCCGCAGGCCGCAGGCCCGAGCGACCTTGCGAGGTGGGAATGTGGATATGACCTGTGCCGAAGCCTCCTAAAACCTCCTAAACCTTCGACGTCGGTTCTCCGCTGGCTTGAGCCGCCAAAACCATTTTGTTCAGATTTCTACGTCACTGAGCGGCACGTGCACCACCAAGGTGGCCCTCCAGGCTGTGGGCCTCTGAGCACGCTGTGGGCCACGTCTCTGCCCACATCCGCCTCAcctccaccccccacacacataaaCTTGTCCTGGGATTATTTAGTCTGTTCCTGGGCTGCTCGCTGGTGGCATTAAACAATGACGTGGGTTTGGGCAAACACACGGCTAAGGTCAGACTCTTCATTCTTCGCTTCCCCCGGCCACTTCCTCTCCCCGAGTGGAGATGGCGGAGGCGGCAGCTGCAGGGTTGGGAAAGAAGGCCAGCGGGAGAGGTCACTGCTCCACCGGAGCCCGCTCCAGGGCGCGCCTCCCCCGTCGTCGCTCCGGCCCCTGAGCTTGGATTTGGAGCCGCAGGAGCACTCGAGGGGCTAAGGGGACAGGAGagctgggaggagccagcagggcTCCGGGCAGGACCAGTGCCCGGGGCAACTTGGAACCAAGCCCCCAGGGCACCAAGGAAGCAGCGGGCAGCCATCCGCGATCGGAGAACAGCGGATCGGCCTGAACCCAGGTCTGCGAGACCAGGAGGCACTGCgctggaggcaggggctgtgAGCCAGGAGTGCGGGACCTGTGGGCTCTGGGAGGTCAGCCCTTCCGAAGGTGCCAGCGCTTCCTAGAATTCTGAGGAGGGGAGAACCTGTGTGTGTTGTTCAGGGCACCGTGAGGTTGCTGGTTGGAGCTGTCAGAGTAGGGGCGGGACacaaggcgggggtgggg"
core_start_input=748
core_end_input=1297
core_start_output=core_start_input-SHIFT_REL_INPUT
core_end_output=core_end_input-SHIFT_REL_INPUT
predict_species_sequence(pre_seq,core_start_input,core_end_input,core_start_output,core_end_output,species,region,show_fig=True)

In [ ]:
#TreeShrew
species="treeshrew"
region="hs3087"
pre_seq="GTCTGGGCACAGGGCTCTCGTCTGTTCGGCCGTGGCGGGGGCCCAGTGCCCAGCGCAAAGTTTGGCACACGGCACTTCCATGGCCTGCTCTGATGAACAAACGCCCCTTCCTAACAGGCCGGGAGCCAGGAGCCGTGGGTCTTGCCCTAACGGCGAGGGCCCCCAGAGAGGGCCTCAGTGCTGTCTCTGGGCCCCCCCTTTCAGTTTGAGGTCAGGCCCGGGGCCGGAGAGTTGGCCTCGTCCGATCACCTCTTTTGGCAGCTCTGGCAGGAGTTGGGGGTCAGGCGAACCTAGGTGTGAGCTGACAGAGCAGTGACCCCCCTGCTGACCTGCATCCCAGGCCCGCAGCCGTCTCCTCCACGTTTTTTGCTATTCTAGGAAAGGCTGTGGCTGGGGAACTTGGGAATGAGGAATCTGACCTTAGATGTGTTCGCCTCGTCACGTCATTGCTTAATGCCACAAACAAGCTGCCGGGGACCAGACTAAATAAGTCCCAAGAGAAGCCCGTGTTGGGGGAGGGTGACCTGCTCGGAGGTCTGCACCCTTGAACCTGACTCATGATGGTGCAGATCCAACCTAATTAAGTTACAGAAATGCTGAATAAAAAATGAGTTCTGCTGCTAAAGCCAGAGAGAGGTTACTTATTTGGGCTCCGACCCTTTCTAACCAATTCCAGACTTGGGATCTGTCTGGAGGTTTTCTCCCAAGACATCTTGCATCCCCTCCAGCCCGAGCTGTCCTCCTTGCAAGGTCACCTTGGCCGGAGTAGGGTGGGTGGCCCAGAGGTCTGTGAGGGACAGGAAGGCCGCTCTGGATGGATTCCTCTGCTGTTGCCACTCGGGGGACGTGATTTTCTTCCCTCCAGACAAGCGCTGGACTTCGTTCTCGACCCCCGCTTGGCATGCCGATGAAGGACAGGATGACAGGCGGCCTGGCCCCATCCCAGTCGGCCCTCTGAGTGGTCCCTAAAATAATACCATTAGTGGGGCTGAGCGGAGGCCCTTATCTGCAGAGCCAGCCTACTGAAGTGCTGAGCGGGGCCCTCGCAGAGCGCACGGCCGGGAGGGGCAGCGGTGTGGGCTGGGGGAGTGGGAgcgggcagggggtgggggctcATAGAGGGCTTCGCTCCCAGCTGGCCTAGGGACATTCCAGGGTATAAATAGCTCGGCAGATGCCTGTCGTGCACCTGGCACGTGATGTCCACCTGGAGCCACGTGTGGAAGCTCTGCCCCTGCCTCTGCTCATGCCAGCCAGGGCCACGGAGGAGCTCGGATGTTGGGGGCACTTGGCCCAGCCTTTGCCCAGCTGGGACCTCCGAGCTGGGCGGGGAGGCTCACCTCAAAGTCATGCCTCCAAGGTTTCTCAGCTGCCCAGACCCCTCActtcctcccccttctccccattcCCCTCTTGCTcgcccttctctattttcttttttagagagttttctgctctagggtacatgcgcttgccggagagacagagagagacagagagacaaaaagtgagaaatcaccctccaagggggtggccacactgcagtcccttctccttcagtctcctctcgggattttggacacgcacacacatgtgctcagtgtgagtttcaagcctccgagcCCGGGACTCACACTGCTGTCTGTGTCAAAgcacagcaatgtaaccatcatgccatctgctggtccctcgctttcttcttttaaaagttgtcttccatgagagagagagagagagacagtaccccagaGGGGCCTCGACGCTGTGACCCACCCCACTCGGCTGCCCTGCATCACGGGCATGCGCATGTACTCGGCACGAACGGGGCCTCCTCGCCAAAGCAGGGTTGTCTCGCGGGACTCTGTACTTTATTGTGAAATAGTACAACTctcagtcctagacccagaatgAAAGAACCGCTTGCGGGGAAGGCGAGCGCTGCCTGGAAGCCGGCCTCCCCTCGCGGTTACAGAACATGAGAACAGCCATGTTTGTTTTGGGGTTCCCCCATCATCCTGCTGCTGCCCCCACGGCCGGCTCCTTCAACTCCGACTGGCCTtctccctgcagccctggcccTCGCCCTGGCCTGCCCCGCGCCATGACCAGCAGCT"
core_start_input=856
core_end_input=1255
core_start_output=core_start_input-SHIFT_REL_INPUT
core_end_output=core_end_input-SHIFT_REL_INPUT
predict_species_sequence(pre_seq,core_start_input,core_end_input,core_start_output,core_end_output,species,region,show_fig=True)

In [ ]:
#TreeShrew_rev--用这个
species="treeshrew"
region="hs3087"
pre_seq="AGCTGCTGGTCATGGCGCGGGGCAGGCCAGGGCGAgggccagggctgcagggagaAGGCCAGTCGGAGTTGAAGGAGCCGGCCGTGGGGGCAGCAGCAGGATGATGGGGGAACCCCAAAACAAACATGGCTGTTCTCATGTTCTGTAACCGCGAGGGGAGGCCGGCTTCCAGGCAGCGCTCGCCTTCCCCGCAAGCGGTTCTTTcattctgggtctaggactgagAGTTGTACTATTTCACAATAAAGTACAGAGTCCCGCGAGACAACCCTGCTTTGGCGAGGAGGCCCCGTTCGTGCCGAGTACATGCGCATGCCCGTGATGCAGGGCAGCCGAGTGGGGTGGGTCACAGCGTCGAGGCCCCtctggggtactgtctctctctctctctctcatggaagacaacttttaaaagaagaaagcgagggaccagcagatggcatgatggttacattgctgtgcTTTGACACAGACAGCAGTGTGAGTCCCGGgctcggaggcttgaaactcacactgagcacatgtgtgtgcgtgtccaaaatcccgagaggagactgaaggagaagggactgcagtgtggccacccccttggagggtgatttctcactttttgtctctctgtctctctctgtctctccggcaagcgcatgtaccctagagcagaaaactctctaaaaaagaaaatagagaagggcgAGCAAGAGGGgaatggggagaagggggaggaagTGAGGGGTCTGGGCAGCTGAGAAACCTTGGAGGCATGACTTTGAGGTGAGCCTCCCCGCCCAGCTCGGAGGTCCCAGCTGGGCAAAGGCTGGGCCAAGTGCCCCCAACATCCGAGCTCCTCCGTGGCCCTGGCTGGCATGAGCAGAGGCAGGGGCAGAGCTTCCACACGTGGCTCCAGGTGGACATCACGTGCCAGGTGCACGACAGGCATCTGCCGAGCTATTTATACCCTGGAATGTCCCTAGGCCAGCTGGGAGCGAAGCCCTCTATgagcccccaccccctgcccgcTCCCACTCCCCCAGCCCACACCGCTGCCCCTCCCGGCCGTGCGCTCTGCGAGGGCCCCGCTCAGCACTTCAGTAGGCTGGCTCTGCAGATAAGGGCCTCCGCTCAGCCCCACTAATGGTATTATTTTAGGGACCACTCAGAGGGCCGACTGGGATGGGGCCAGGCCGCCTGTCATCCTGTCCTTCATCGGCATGCCAAGCGGGGGTCGAGAACGAAGTCCAGCGCTTGTCTGGAGGGAAGAAAATCACGTCCCCCGAGTGGCAACAGCAGAGGAATCCATCCAGAGCGGCCTTCCTGTCCCTCACAGACCTCTGGGCCACCCACCCTACTCCGGCCAAGGTGACCTTGCAAGGAGGACAGCTCGGGCTGGAGGGGATGCAAGATGTCTTGGGAGAAAACCTCCAGACAGATCCCAAGTCTGGAATTGGTTAGAAAGGGTCGGAGCCCAAATAAGTAACCTCTCTCTGGCTTTAGCAGCAGAACTCATTTTTTATTCAGCATTTCTGTAACTTAATTAGGTTGGATCTGCACCATCATGAGTCAGGTTCAAGGGTGCAGACCTCCGAGCAGGTCACCCTCCCCCAACACGGGCTTCTCTTGGGACTTATTTAGTCTGGTCCCCGGCAGCTTGTTTGTGGCATTAAGCAATGACGTGACGAGGCGAACACATCTAAGGTCAGATTCCTCATTCCCAAGTTCCCCAGCCACAGCCTTTCCTAGAATAGCAAAAAACGTGGAGGAGACGGCTGCGGGCCTGGGATGCAGGTCAGCAGGGGGGTCACTGCTCTGTCAGCTCACACCTAGGTTCGCCTGACCCCCAACTCCTGCCAGAGCTGCCAAAAGAGGTGATCGGACGAGGCCAACTCTCCGGCCCCGGGCCTGACCTCAAACTGAAAGGGGGGGCCCAGAGACAGCACTGAGGCCCTCTCTGGGGGCCCTCGCCGTTAGGGCAAGACCCACGGCTCCTGGCTCCCGGCCTGTTAGGAAGGGGCGTTTGTTCATCAGAGCAGGCCATGGAAGTGCCGTGTGCCAAACTTTGCGCTGGGCACTGGGCCCCCGCCACGGCCGAACAGACGAGAGCCCTGTGCCCAGAC"
core_start_input=834
core_end_input=1383
core_start_output=core_start_input-SHIFT_REL_INPUT
core_end_output=core_end_input-SHIFT_REL_INPUT
predict_species_sequence(pre_seq,core_start_input,core_end_input,core_start_output,core_end_output,species,region,show_fig=True)

In [ ]:
pre_seq="aaagctcagtggcggtgccaagggctgtgggatttcctttgcagctctattcacccctttcagctccacccacaatgggaacagacacttctggatgaaataaaaccaaagactcatccttgaagttcagagtttgcatcctctgcatgtttccatccagatacccctgcctgctgacttgcatagctgccatgaacacgttattcactgtattcagccgttcctttggagaaaaatactgcccaattaagttaaaaggagcccgtttgaagtttttatttatcaggtaatgattgacaagagcttgaggccaaaactaggagcactttatactataatcattcgttacccaaggacttaagcaacacaaatgggttgttaaggccactacgagttactctattctaaataaaaatattgattcccttttgcttaatacagtaaaatttgtattcccacaccagcaggaaagaaagatcagaagttcttgataacacaatggtttgttccccaatacgcacattgctgtttccccaggggcttctgaccaggcactacgagaggactgagtttaatgctttgagacttattttgatactgtgaggtttgccacctgacatttgaattagttccatacatgaagagagtacagcctttagacagcttgactttggttttatatttatctgcttcacttttttccccccttccatcagaggaaagtactgggggagcaaggggagaacaggatgctatggtttcagtcccattaaaccttagaaaggtagcttcatgcttggctaattgttcagactgtttctgcaggcacagttgagcatttaaccatttgcaggataaagctcaatacagagctgcttgaattaggctcaccctcaaagaattggttttgcttgtctgcagtaaaacccaatcccaagccagccagcaataatttggggaacagagcatttctgtctggagctgaggggtaagtggctgtattcactgctacatccacgtaattagaaacatgcacttaccctttccgaattagaagctgaagggactctgcagttaacagctctgtgctgctgataacgtgggcctaatgaccttattttaggtagctcttaagacaaggcaggggttcccagccccccggcatatctgtcatgctttagtgcatcagcacccggaactcaagccaaggatttcaggctgatgctaaacttccaaagcagcatatacaggtttgggaagtcacagtggttgcagggtacaaaaagcctcattagtggcatcaggagccttttatcccttcatctccatcccatgagctctccacaacagtgggcaatggtcctggcagtggcctgacactagcagtgacaccagcagggcgacctctgtcttggctaatggatgcaaactgttccaggccactggatttattcctctgctatttgtcactgctgtgtcaggtgcttcatataaaatacatggaagcagagagatgcagtgtctggattctgttaggaagggtcagagctcaaagatatcacttctcaagtcccagcaacgtaagccatcttgtttggcgtttgtaacaagctgtagacaaaagcagagaggtcgcaggctctacatctgtctcacttgtgtgcaagcactactcagcctctcagaaaagatacttgagaacaaattccttattttagaatccagcttctactgcatgctgtgcacacagggagggagcacaaatcagcacactcctctaattcaaaacggactcaaagctttgcttatgattagttattaatttctcagtgaccttctgctaacactgaccaatatttggatcctgaacctaaaatctttcactaaaaatccagtgaactgcattgttcttccttaacagggagtgacctggcctgactgcagttctaagacttggagataattattccaaggccttattttacaaaacctcggggctggaactctaagcagacagtaactgggggtactgcaaagctcctcacctgacagctgtacaatcacctgttgaaatccatcttttaaggtttctcttgcaaacctggccagcaattctggaggaggagtaactcaacagtcatggaggttggacctccattggcttggagaaattttagccaagcaatgtttatcctaactgcaaggcagacattccctatgccctctttctttttttaaataaaagaaaattggagtcatacacaaatgattctaagttcagtccatcagttgttttggtttcacatgactactaagaaatcaactgtaaaagctagtgcgataatatacacacataccatgcacagaatgtgagctataaatattggcagagggtctggttcaagaagctgcagttaatgttgtgttagagcagcccttggaacccccatttccagagatttatactgcagggggtggggagggaataaatccgattttgctggaacattgcctgcaacttctcaaccataaagcctttatttatggctgttacacagcatgaaggattcaatttgtcctttgttatgaatttcagtggatcataactgtttatttccagatttttattttaaagttggcatctttggtttgcaggaagacaaagctcttcataactcattccagtggggatgtgctcagaccacactcaacagagatagtatagcatgatcccagagtccagcaatatctaccgggtccagaggtaagggcagtactaatggacatttatgaggtcaacctgatggggatgaggggatgaaggctaggctccaggctgaagtacattca"
len(pre_seq)

In [ ]:
#chick-1
species="chick"
region="hs3087-1"
pre_seq="aaagctcagtggcggtgccaagggctgtgggatttcctttgcagctctattcacccctttcagctccacccacaatgggaacagacacttctggatgaaataaaaccaaagactcatccttgaagttcagagtttgcatcctctgcatgtttccatccagatacccctgcctgctgacttgcatagctgccatgaacacgttattcactgtattcagccgttcctttggagaaaaatactgcccaattaagttaaaaggagcccgtttgaagtttttatttatcaggtaatgattgacaagagcttgaggccaaaactaggagcactttatactataatcattcgttacccaaggacttaagcaacacaaatgggttgttaaggccactacgagttactctattctaaataaaaatattgattcccttttgcttaatacagtaaaatttgtattcccacaccagcaggaaagaaagatcagaagttcttgataacacaatggtttgttccccaatacgcacattgctgtttccccaggggcttctgaccaggcactacgagaggactgagtttaatgctttgagacttattttgatactgtgaggtttgccacctgacatttgaattagttccatacatgaagagagtacagcctttagacagcttgactttggttttatatttatctgcttcacttttttccccccttccatcagaggaaagtactgggggagcaaggggagaacaggatgctatggtttcagtcccattaaaccttagaaaggtagcttcatgcttggctaattgttcagactgtttctgcaggcacagttgagcatttaaccatttgcaggataaagctcaatacagagctgcttgaattaggctcaccctcaaagaattggttttgcttgtctgcagtaaaacccaatcccaagccagccagcaataatttggggaacagagcatttctgtctggagctgaggggtaagtggctgtattcactgctacatccacgtaattagaaacatgcacttaccctttccgaattagaagctgaagggactctgcagttaacagctctgtgctgctgataacgtgggcctaatgaccttattttaggtagctcttaagacaaggcaggggttcccagccccccggcatatctgtcatgctttagtgcatcagcacccggaactcaagccaaggatttcaggctgatgctaaacttccaaagcagcatatacaggtttgggaagtcacagtggttgcagggtacaaaaagcctcattagtggcatcaggagccttttatcccttcatctccatcccatgagctctccacaacagtgggcaatggtcctggcagtggcctgacactagcagtgacaccagcagggcgacctctgtcttggctaatggatgcaaactgttccaggccactggatttattcctctgctatttgtcactgctgtgtcaggtgcttcatataaaatacatggaagcagagagatgcagtgtctggattctgttaggaagggtcagagctcaaagatatcacttctcaagtcccagcaacgtaagccatcttgtttggcgtttgtaacaagctgtagacaaaagcagagaggtcgcaggctctacatctgtctcacttgtgtgcaagcactactcagcctctcagaaaagatacttgagaacaaattccttattttagaatccagcttctactgcatgctgtgcacacagggagggagcacaaatcagcacactcctctaattcaaaacggactcaaagctttgcttatgattagttattaatttctcagtgaccttctgctaacactgaccaatatttggatcctgaacctaaaatctttcactaaaaatccagtgaactgcattgttcttccttaacagggagtgacctggcctgactgcagttctaagacttggagataattattccaaggccttattttacaaaacctcggggctggaactctaagcagacagtaactgggggtactgcaaagctcctcacctgacagctgtacaatcacctgttgaaatccatcttttaaggtttctcttgcaaacctggccagcaattctggaggaggagtaactcaacagtcatggaggttggacctccattggcttggagaaattttagccaagcaatgtttatcctaactgcaaggcagacattccctatgccctctttctttttttaaataaaagaaaattggagtcatacacaaatgattctaagttcagtccatcagttgttttggtttcacatgactactaagaaatcaactgtaaaagctagtgcgataatatacacacataccatgcacagaatgtgagctataaatattggcagagggtctggttcaagaagctgcagttaatgttgtgttagagcagcccttggaacccccatttccagagatttatactgcagggggtggggagggaataaatccgattttgctggaacattgcctgcaacttctcaaccataaagcctttatttatggctgttacacagcatgaaggattcaatttgtcctttgttatgaatttcagtggatcataactgtttatttccagatttttattttaaagttggcatctttggtttgcaggaagacaaagctcttcataactcattccagtggggatgtgctcagaccacactcaacagagatagtatagcatgatcccagagtccagcaatatctaccgggtccagaggtaagggcagtactaatggacatttatgaggtcaacctgatggggatgaggggatgaaggctaggctccaggctgaagtacattca"
core_start_input=807
core_end_input=1307
core_start_output=core_start_input-SHIFT_REL_INPUT
core_end_output=core_end_input-SHIFT_REL_INPUT
predict_species_sequence(pre_seq,core_start_input,core_end_input,core_start_output,core_end_output,species,region,show_fig=True)

In [ ]:
#chick-2
species="chick"
region="hs3087-2"
pre_seq="tgggacagcttccaccagcaccatgggaactgtgtttctaagctctggcctcaggaaatcctggtttccaggaagaaaagtccctttcatcttagaggctgggccttcccaatagtttaaaccaagagggccacataatcctttttcttcttctgtgaggatgtttgtaacccagggacaatgtgtattccacagcagccacttcagaattccggggggagaaccaagtgatgctgttatttgttattaccaacagaactgcaaggaggtcctgaagggctgagcaggaggggaggaccaaactccttcacacagctcccactgctgctcctcctcacatcattgttgtgcttcagaagcaggaagaaatgtttcaatctcatgttgttgtgatcaccatatctcccctatggcgcaaatcttaagatcctttgaactttcaaaagcacatgtgccttttcttgttgctggagctaaagctaatatcagaggcagcacttctggagcatcagaatgaacatttcactctttggtgatgaggaatcatttaccaaatgccttaacaaattgaactaagccaaacagttcttacttcatgaagacatgaacttcaccctgctgacaaagtgaggttgcaatagtttgggttatttgtgtttgctgcacagaaatgctacctggcggcaccctgagcccactgggactcgttccatggggtctgtcctgccagcctctgtgtccctctggccaccgcgtgctgcttgtcacagagaggccagcaggtacctcggctcctctgccacagcaagggagcattcccattcataaaggatgaacccaacatccttctccctcccggggaaaccctggacaaaagcagaaaagccgcagagcttcccctcaaagcccacccccagcaatgctccggacgcttccacggcgtttacccatcccgctgcgctgcgctgcgctgttcggaacgcgattcgcgcacactgcggccgcgctccgcgccggcagccgcctgggacggcactgctgacctctgccggcagcgcacggcgagacccgagagtggcggccacagagggggaagcgggggggtgtgggggggtgtggtaccgctgtcccttacgcagtgccagggcagcaccgcaggagcagcgcaggctgtcagcctttagccacgcgtttgttacttcctctagaagaagatttttgcagctactccactttggcaacccaaggcaacttctgtgacgcttctgtgctatgaacaacaccgctttacctctcttcctaacctttgccatccaactgatgcaatatctgacacctgatgtcctttgaggagacctcattgcaccttctagtacctaaagggaccttatcagcaggagaggaaccgactttttacattatcataatgatgggacaagggggaaaggctttaaactgaaagaagggaggtttaggttagatgtgagggcggtgaggcgctgcccagagctgtggtgctccatccctgctgtgcccgaggctgtggatggatactgggcagcctgagctggggggcagccaggtcacggcaagggggaggggggtgaggagcatcagatcccttccaatccaaccacgctatgattctgtggttctgcgattctgtgaaatcatttttgagcgatcttttttcttctattattggactgaaataggctgaagccttaaatacctgctgtgcatttctttcacgggcggtagggggcgcgatgaaaatgggggagagggccgagaaggggaaccgcatctcctcccgcagcggcgggcagggggagctcacgctgcgctgatcgcggccctttgcgccgtcctcagctgcaacttttcggcgttccgtccgtctctgcaacagaagtaacgcgatctgctgttgcggggcacacgcagcactgcctacagcagcctaagaacgagcagagctcggcccggtgcaggacgagccaaaccatctgacgtgcggcgttcttcactgcaattactttgggagcagcatctgggcccacgtgagcctgagctcccttcaggtgtgtccctgcaccacctttgctgtgagcatcgctcctgaagagatgggagagggctcctatgggcccaaagtccacggccttccctgtttctgtatttgatttcagcaacaagcaacgtgaaccattcctggagtgcagaggaacttcaccctaacactgccgctctgacatctcggccgttacg"
core_start_input=807
core_end_input=1307
core_start_output=core_start_input-SHIFT_REL_INPUT
core_end_output=core_end_input-SHIFT_REL_INPUT
predict_species_sequence(pre_seq,core_start_input,core_end_input,core_start_output,core_end_output,species,region,show_fig=True)

In [ ]:
#chick-3
species="chick"
region="hs3087-3"
pre_seq="gcatcagaatgaacatttcactctttggtgatgaggaatcatttaccaaatgccttaacaaattgaactaagccaaacagttcttacttcatgaagacatgaacttcaccctgctgacaaagtgaggttgcaatagtttgggttatttgtgtttgctgcacagaaatgctacctggcggcaccctgagcccactgggactcgttccatggggtctgtcctgccagcctctgtgtccctctggccaccgcgtgctgcttgtcacagagaggccagcaggtacctcggctcctctgccacagcaagggagcattcccattcataaaggatgaacccaacatccttctccctcccggggaaaccctggacaaaagcagaaaagccgcagagcttcccctcaaagcccacccccagcaatgctccggacgcttccacggcgtttacccatcccgctgcgctgcgctgcgctgttcggaacgcgattcgcgcacactgcggccgcgctccgcgccggcagccgcctgggacggcactgctgacctctgccggcagcgcacggcgagacccgagagtggcggccacagagggggaagcgggggggtgtgggggggtgtggtaccgctgtcccttacgcagtgccagggcagcaccgcaggagcagcgcaggctgtcagcctttagccacgcgtttgttacttcctctagaagaagatttttgcagctactccactttggcaacccaaggcaacttctgtgacgcttctgtgctatgaacaacaccgctttacctctcttcctaacctttgccatccaactgatgcaatatctgacacctgatgtcctttgaggagacctcattgcaccttctagtacctaaagggaccttatcagcaggagaggaaccgactttttacattatcataatgatgggacaagggggaaaggctttaaactgaaagaagggaggtttaggttagatgtgagggcggtgaggcgctgcccagagctgtggtgctccatccctgctgtgcccgaggctgtggatggatactgggcagcctgagctggggggcagccaggtcacggcaagggggaggggggtgaggagcatcagatcccttccaatccaaccacgctatgattctgtggttctgcgattctgtgaaatcatttttgagcgatcttttttcttctattattggactgaaataggctgaagccttaaatacctgctgtgcatttctttcacgggcggtagggggcgcgatgaaaatgggggagagggccgagaaggggaaccgcatctcctcccgcagcggcgggcagggggagctcacgctgcgctgatcgcggccctttgcgccgtcctcagctgcaacttttcggcgttccgtccgtctctgcaacagaagtaacgcgatctgctgttgcggggcacacgcagcactgcctacagcagcctaagaacgagcagagctcggcccggtgcaggacgagccaaaccatctgacgtgcggcgttcttcactgcaattactttgggagcagcatctgggcccacgtgagcctgagctcccttcaggtgtgtccctgcaccacctttgctgtgagcatcgctcctgaagagatgggagagggctcctatgggcccaaagtccacggccttccctgtttctgtatttgatttcagcaacaagcaacgtgaaccattcctggagtgcagaggaacttcaccctaacactgccgctctgacatctcggccgttacgaagggctctgcacgttttcctgctcccttgttctaacagaggtggctgggcagcacttctcttgacacacacgcggttgtgtttgctcagaactcactcctgggggctggcagatatgccagaaggttcctgaggatccttacaagccattcctgagttataacagcacacggatcagttatctcacagggtacctttccaaacacatcctttagaacagctgcgatggtgctgagttagcatttccaaaatattaacattcctattgtagcagtgcagaagcttaattgctgagcaagtataaacaaggagttccagttacaaaattctcgcaaacacccaggattcatctgcttttttttcttttttaattacattttattgccttcattttgtaaacgccaaaaatacaatggctcccattatgaagttctgtgttgtagttccccacccaggacgatgtgcggcagtctttgtgagctctcaggcccatctctgatgtgctgcatgccaaggctgggagatcctcgctatttcacacatttgcagtatgtcaccacaacaccaccgcccaacccgtatgtaggttttctgcacaaaaatgtcttcacttgctcatttctcaagactatctcagttttccatcccaaactgagatgctccacccagtcgtagcacacacaaagattattccaagcctggggcagaggaaagagctgaatgtgcaccagcataagccatgctggctgtgcccacaagaccgcagcatcacagtgggggcacgtgcacaggatgaggccctgtgccccgttctgctctttgaaagctgcacatcatttctgctatcagaagctttcggcttcctgctgtgctgaatgacagtggctgaaaggggatcctgcgttcagcaaggcagctcccctgggcccgtggtttactgctctccagaggtgcagacgatgtgatgcctgagctgccttaggtgggagggacagccccggccacccctgccctgctcagcaccacgtcctgagtgatgcttctcccgtaaccacacgcctggcagggagcgaaagggaccagatctcagcagcagctgcaccaatcccagggctgttctgtgccctccatcctccacaactgcctccagccccactgagagccaccacagagcccttccctcagatgcgctccgctttccctccacctctttcacatatcaagtcaagctttccctccccatttccaaacctcttcatcacagttcccc"
core_start_input=807
core_end_input=1307
core_start_output=core_start_input-SHIFT_REL_INPUT
core_end_output=core_end_input-SHIFT_REL_INPUT
predict_species_sequence(pre_seq,core_start_input,core_end_input,core_start_output,core_end_output,species,region,show_fig=True)

In [ ]:
#human-short
#species="human"
#region="hs3087"
#pre_seq="accctgcatgaagacgtccacgtggccagaaaggcaggctctgccggccaaagcgaggagagagtgctcacgattcattataaaagtgaggaagtgtcctctgagcacgcctccaaggcggttctgcgcagctcctgctggggcagctgagagcgaacgatgtgctgagtgaactcctgaggttcagcaaacctgcgcaaaggttccagggacccatcttctgaggggcttctcctgcaggtggtccaggcagccatcaaagacggggctgcaggggggtctggggcccaggatgacatttgttattcagtaagtgaagagagaaggctacaaaaattgagtataatcccttttgtattaaaacaacagcaaaacaatgtaagtatggatagaaaaaaagtctaaaaataagtatagctaatatccagtggaaggagaaggtcagatttaaggacaatttttttttttttaccttttcctgtgtgccgttctttatttccaaatctaggaccaagaataataaatgtgaacaaaagaagcattctgatgatgaactgtgggcagccacagaaatgcttaaggcacgatattgaaggtgagcacccttcaatatccctgtccctgaagcagggacaaggaggtcacggctgggcaaggctgggaggaggagttcacagggcccactctggctgaggacatggcggcagaggcagcagcaggtgcctgtgcaccgtggctttggttgggtgtcacataccaggtgtgccaggctctgccgctgccgccatgcctgcacaatcagacgtggagcccgtcggactatttataccccttggaacatcagcaggccagccaggaggcgaagaccttcgcaacatgtccaaggcccgcagctgctacccctcctcacccagggcttccgtgaggacctgctcagtgttatcagtgcacccactccacagataagaagctgtgcctggccccactaatggtattattttagggaccactcaaaggacaggccaggattggggcctgggtcatctgtcattccttccttcatcgacacgctaaataagggtcaaacagagaccagcacttgtttgcagggctggaggggctctgccaagctggagtatcatcctccccagagactgctgggaaaggcatttgctcagagcagctatttctgtcccactcacagcacctctgggtgacccacccacaggtcagagtgaccttgcaaggtggaaggacaaagacattgctgcctaactcttggtgagagggaatgtgtaagatggcttatgctaaaacctcccaaaatagagataccacagatctggattctgttacaaagggtcagagctcaaatatgtaatttctctgctggctttagcattcgaaaccgtttttactcagcatttcttttttctttctttctttttttttctttcgagacagagtttcgctcttgttgcccaggctggagtgcaacggtgcaatcttggctcaccgcaacccttctgcctcccgggttcaagtgattctgctgcccctgcctcctgagtagctggcattacaggcatgtgccaccatgcccggctaattttgtagttttagtagagatgggggttctccatgttggtcaggctggtctccagctcccaacctcaggtgatcagcccacctcggcctcccaaagtgctgggattataggcatgagccaccacacctggcccagcatttctataatttaattgtgttagatatgcaccaccgagggtcaagtacaaggatgtggaccttggagcaaattgtggccaaagtcaccacccaccatcagtctccaccttcgcccccacagataccccctgccaacacacacacatgcacgcacacacacacacacatatataaacttttcttgggaattatttagtctgttactcagtgacttgcctgtggcattaaacaatgatgtgagtttgagcaaacatgcatccaaggtcagattcatcacacgtaatgtccttcctaaagccacttccttttccagaacgaaaaaaagagcagaggaaacagctacagggtttgggatgagggtcagcagg"
#core_start_input=807
#core_end_input=1256
#core_start_output=core_start_input-SHIFT_REL_INPUT
#core_end_output=core_end_input-SHIFT_REL_INPUT
#predict_species_sequence(pre_seq,core_start_input,core_end_input,core_start_output,core_end_output,species,region,show_fig=True)

#human-long
species="human"
region="hs3087"
pre_seq="accctgcatgaagacgtccacgtggccagaaaggcaggctctgccggccaaagcgaggagagagtgctcacgattcattataaaagtgaggaagtgtcctctgagcacgcctccaaggcggttctgcgcagctcctgctggggcagctgagagcgaacgatgtgctgagtgaactcctgaggttcagcaaacctgcgcaaaggttccagggacccatcttctgaggggcttctcctgcaggtggtccaggcagccatcaaagacggggctgcaggggggtctggggcccaggatgacatttgttattcagtaagtgaagagagaaggctacaaaaattgagtataatcccttttgtattaaaacaacagcaaaacaatgtaagtatggatagaaaaaaagtctaaaaataagtatagctaatatccagtggaaggagaaggtcagatttaaggacaatttttttttttttaccttttcctgtgtgccgttctttatttccaaatctaggaccaagaataataaatgtgaacaaaagaagcattctgatgatgaactgtgggcagccacagaaatgcttaaggcacgatattgaaggtgagcacccttcaatatccctgtccctgaagcagggacaaggaggtcacggctgggcaaggctgggaggaggagttcacagggcccactctggctgaggacatggcggcagaggcagcagcaggtgcctgtgcaccgtggctttggttgggtgtcacataccaggtgtgccaggctctgccgctgccgccatgcctgcacaatcagacgtggagcccgtcggactatttataccccttggaacatcagcaggccagccaggaggcgaagaccttcgcaacatgtccaaggcccgcagctgctacccctcctcacccagggcttccgtgaggacctgctcagtgttatcagtgcacccactccacagataagaagctgtgcctggccccactaatggtattattttagggaccactcaaaggacaggccaggattggggcctgggtcatctgtcattccttccttcatcgacacgctaaataagggtcaaacagagaccagcacttgtttgcagggctggaggggctctgccaagctggagtatcatcctccccagagactgctgggaaaggcatttgctcagagcagctatttctgtcccactcacagcacctctgggtgacccacccacaggtcagagtgaccttgcaaggtggaaggacaaagacattgctgcctaactcttggtgagagggaatgtgtaagatggcttatgctaaaacctcccaaaatagagataccacagatctggattctgttacaaagggtcagagctcaaatatgtaatttctctgctggctttagcattcgaaaccgtttttactcagcatttcttttttctttctttctttttttttctttcgagacagagtttcgctcttgttgcccaggctggagtgcaacggtgcaatcttggctcaccgcaacccttctgcctcccgggttcaagtgattctgctgcccctgcctcctgagtagctggcattacaggcatgtgccaccatgcccggctaattttgtagttttagtagagatgggggttctccatgttggtcaggctggtctccagctcccaacctcaggtgatcagcccacctcggcctcccaaagtgctgggattataggcatgagccaccacacctggcccagcatttctataatttaattgtgttagatatgcaccaccgagggtcaagtacaaggatgtggaccttggagcaaattgtggccaaagtcaccacccaccatcagtctccaccttcgcccccacagataccccctgccaacacacacacatgcacgcacacacacacacacatatataaacttttcttgggaattatttagtctgttactcagtgacttgcctgtggcattaaacaatgatgtgagtttgagcaaacatgcatccaaggtcagattcatcacacgtaatgtccttcctaaagccacttccttttccagaacgaaaaaaagagcagaggaaacagctacagggtttgggatgagggtcagcagg"
core_start_input=707
core_end_input=1256
core_start_output=core_start_input-SHIFT_REL_INPUT
core_end_output=core_end_input-SHIFT_REL_INPUT
predict_species_sequence(pre_seq,core_start_input,core_end_input,core_start_output,core_end_output,species,region,show_fig=True)
